In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

years = range(2019, 2025)

temp = (
    pd.read_csv("../../data/07_portfolios_metadata/all_tickers_complete_metadata.csv")
      .replace({"#ERROR!": np.nan})
)

centralities = ["central", "peripheral"]
modes = ["fast", "medium", "slow"]

k_values = [1, 10]

beta_momentum = pd.read_csv("../../data/07_portfolios_metadata/beta_momentum.csv", index_col="Unnamed: 0")

for k in k_values:
    all_years_df = []

    for year in years:
        print(f"Processing year {year}, k={k}")

        port_dict = {}

        for centrality in centralities:
            port_dict[f"{centrality}"] = (
                pd.read_csv(
                    f"../../data/07_portfolios_metadata/{centrality}_metadata_{year}_{k}.csv"
                )[["Ticker", "Country"]]
                .merge(temp, on="Ticker", how="inner")
            )

        dfs = []
        for portfolio_name, df in port_dict.items():
            df = df.copy()
            df["portfolio"] = portfolio_name.replace("_", " ")
            df["year"] = year
            dfs.append(df)

        final_df = pd.concat(dfs, ignore_index=True)

        final_df = final_df.merge(beta_momentum, on="Ticker", how="left")

        cols_to_clean = [
            col for col in final_df.columns
            if str(year) in col
        ]

        final_df[cols_to_clean] = (
            final_df[cols_to_clean]
                .apply(lambda s: s.astype(str).str.replace(",", ".", regex=False))
                .apply(pd.to_numeric, errors="coerce")
        )

        cols_to_keep = [
            "Ticker", "Country", "Company", "Sector",
            "Industry", "portfolio", "year"
        ] + cols_to_clean

        final_df = final_df[cols_to_keep]

        for col in cols_to_clean:
            final_df[f"{col}_cut"] = (
                pd.qcut(
                    final_df[col],
                    q=2,
                    labels=False,
                    duplicates="drop"
                ) + 1
            )

        all_years_df.append(final_df)

    final_panel_df = pd.concat(all_years_df, ignore_index=True)
    final_panel_df.to_csv(f"../../data/07_portfolios_metadata/complete_metadata_{k}.csv", index=False)
    print(f"Saved complete_metadata_{k}.csv")

Processing year 2019, k=1
Processing year 2020, k=1
Processing year 2021, k=1
Processing year 2022, k=1
Processing year 2023, k=1
Processing year 2024, k=1
Saved complete_metadata_1.csv
Processing year 2019, k=10
Processing year 2020, k=10
Processing year 2021, k=10
Processing year 2022, k=10
Processing year 2023, k=10
Processing year 2024, k=10
Saved complete_metadata_10.csv


In [11]:
final_panel_df.query("portfolio=='central' and year==2019").shape

(80, 163)

In [12]:
final_panel_df.to_csv("../../data/07_portfolios_metadata/complete_metadata.csv")